In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!mkdir -p /content/contract-reviewer/data
!unzip -q -o /content/drive/MyDrive/CUAD_v1.zip -d /content/contract-reviewer/data
!unzip -q -o /content/drive/MyDrive/code.zip -d /content/contract-reviewer
!pip install -q sentence-transformers rank_bm25

%cd /content/contract-reviewer
!mkdir -p notebooks /content/drive/MyDrive/contract-cache
!ln -s /content/drive/MyDrive/contract-cache data/cache
%cd notebooks

import torch
print("GPU:", torch.cuda.is_available())
!ls ..

/content/contract-reviewer
/content/contract-reviewer/notebooks
GPU: True
data  eval  notebooks  src


In [ ]:
import sys, time
sys.path.append("..")
from functools import partial
import pandas as pd
from src.data import load_cuad, TARGET_CLAUSES
from src.split import split_contracts
from src.chunking import fixed_chunks, clause_chunks
from src.retrieval import make_dense_ranker
from src.metrics import evaluate_ranker

contracts = load_cuad()
dev_idx, test_idx = split_contracts(contracts)
print(len(contracts), len(dev_idx), len(test_idx))

510 357 153


In [1]:
r = make_dense_ranker(clause_chunks, tag="speedtest")
t = time.time()
evaluate_ranker(r, contracts, dev_idx[:10], "Governing Law")
print(round(time.time() - t, 1), "seconds for 10 contracts")

NameError: name 'make_dense_ranker' is not defined

In [ ]:
configs = {
    "fixed1000":   (fixed_chunks, "fixed-1000-200"),
    "clause1000":  (partial(clause_chunks, max_size=1000), "clause-200-1000-200"),
    "clause1500":  (partial(clause_chunks, max_size=1500), "clause-200-1500-200"),
    "clause2000":  (clause_chunks, "clause-200-2000-200"),
}

results = {}
for name, (fn, tag) in configs.items():
    r = make_dense_ranker(fn, tag=tag)
    rows = {cl: evaluate_ranker(r, contracts, dev_idx, cl) for cl in TARGET_CLAUSES}
    df = pd.DataFrame(rows).T
    df.to_csv(f"../eval/results_colab_{name}_dev.csv")
    results[name] = df
    print("done:", name)

done: fixed1000
done: clause1000
done: clause1500
done: clause2000


In [ ]:
summary = pd.DataFrame(
    {k: v[["hit@1", "hit@3", "mrr"]].mean() for k, v in results.items()}
).T.round(3)
summary

,hit@1,hit@3,mrr
fixed1000,0.566,0.772,0.690
clause1000,0.591,0.796,0.710
clause1500,0.611,0.800,0.723
clause2000,0.600,0.795,0.715


In [ ]:
!cp ../eval/results_colab_*_dev.csv /content/drive/MyDrive/